In [2]:
from pyspark.sql import SparkSession
import os
print (os.getcwd())

/home/siribe


In [25]:
spark = SparkSession.builder \
    .appName("musicbrainz") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.instances", "1") \
    .getOrCreate()

Recording Data Initial Exploration

In [4]:
recordingdf = spark.read.json("/home/siribe/musicbrainz_json/mbdump/recording")
recordingdf.printSchema()
recordingdf.count()

root
 |-- aliases: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- begin: string (nullable = true)
 |    |    |-- end: string (nullable = true)
 |    |    |-- ended: boolean (nullable = true)
 |    |    |-- locale: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- primary: boolean (nullable = true)
 |    |    |-- sort-name: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |    |    |-- type-id: string (nullable = true)
 |-- annotation: string (nullable = true)
 |-- artist-credit: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- artist: struct (nullable = true)
 |    |    |    |-- aliases: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- begin: string (nullable = true)
 |    |    |    |    |    |-- end: string (nullable = true)
 |    |    |    |    |    |-- ended: boolean (nullable = tr

148797

In [6]:
recordingdf.describe().show()

+-------+--------------------+------------------+--------------------+------------------+--------------------+
|summary|          annotation|    disambiguation|                  id|            length|               title|
+-------+--------------------+------------------+--------------------+------------------+--------------------+
|  count|                3145|            148797|              148797|            107657|              148797|
|   mean|              2007.0|1942.8783018867925|                NULL|335967.36223376094|            Infinity|
| stddev|                NULL| 326.1514378043592|                NULL|1661094.8331819875|                 NaN|
|    min|                    |                  |00003931-e8a9-411...|                 1|! ! JUMP IN2 the ...|
|    max|🎡ViDEO\nDirector...|                🩸|ffffad8e-7b64-4dd...|         227710832|              🥺👉👈|
+-------+--------------------+------------------+--------------------+------------------+--------------------+



In [13]:
recordingdf.columns

['aliases',
 'annotation',
 'artist-credit',
 'disambiguation',
 'genres',
 'id',
 'isrcs',
 'length',
 'rating',
 'relations',
 'tags',
 'title',
 'video']

In [24]:
recordingdf.groupBy("genres") \
    .agg(count("*").alias("num_genres")) \
    .show()

+--------------------+----------+
|              genres|num_genres|
+--------------------+----------+
|[{1, , a715278f-1...|        29|
|[{2, , e5bba957-8...|         1|
|[{1, , 4f9a1d5e-e...|         1|
|[{1, , 8cc9b280-2...|        14|
|[{1, , 6cec5b4d-3...|         1|
|[{1, , 462f9321-6...|         1|
|[{1, , 911c7bbb-1...|         9|
|[{1, , ceeaa283-5...|         2|
|[{2, , 93244085-2...|         1|
|[{1, , b7864789-2...|         2|
|[{1, , ceeaa283-5...|         3|
|[{1, , 2afc2320-3...|        58|
|[{1, , 93244085-2...|         2|
|[{1, , 06eeec6e-f...|         1|
|[{1, , 2c765e49-a...|         1|
|[{1, , 060beed7-e...|         2|
|[{1, , 3832a068-e...|         1|
|[{1, , 7fa479e7-3...|         7|
|[{1, , b739a895-8...|         1|
|[{1, , ceeaa283-5...|         1|
+--------------------+----------+
only showing top 20 rows



Instrument Data Exploration

In [5]:
instrumentdf = spark.read.json("/home/siribe/musicbrainz_json/mbdump/instrument")
instrumentdf.printSchema()
instrumentdf.count()

root
 |-- aliases: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- begin: string (nullable = true)
 |    |    |-- end: string (nullable = true)
 |    |    |-- ended: boolean (nullable = true)
 |    |    |-- locale: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- primary: boolean (nullable = true)
 |    |    |-- sort-name: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |    |    |-- type-id: string (nullable = true)
 |-- annotation: string (nullable = true)
 |-- description: string (nullable = true)
 |-- disambiguation: string (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- count: long (nullable = true)
 |    |    |-- disambiguation: string (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |

1056

In [8]:
instrumentdf.describe().show()

+-------+--------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+
|summary|          annotation|         description|      disambiguation|                  id|            name|                type|             type-id|
+-------+--------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+
|  count|                  46|                1056|                1056|                1056|            1056|                1055|                1055|
|   mean|                NULL|                NULL|                NULL|                NULL|            NULL|                NULL|                NULL|
| stddev|                NULL|                NULL|                NULL|                NULL|            NULL|                NULL|                NULL|
|    min|\nThis instrument...|                    |                    |007d6c88-c

In [12]:
instrumentdf.columns

['aliases',
 'annotation',
 'description',
 'disambiguation',
 'genres',
 'id',
 'name',
 'relations',
 'tags',
 'type',
 'type-id']

In [18]:
from pyspark.sql.functions import count

instrumentdf.groupBy("type") \
    .agg(count("*").alias("num_instruments")) \
    .show()

+--------------------+---------------+
|                type|num_instruments|
+--------------------+---------------+
|     Wind instrument|            312|
|                NULL|              1|
|    Other instrument|             17|
|Electronic instru...|             51|
|              Family|             16|
|Percussion instru...|            287|
|   String instrument|            352|
|            Ensemble|             20|
+--------------------+---------------+

